In [ ]:
#!pip install arxiv transformers

In [ ]:
import arxiv
import os
from pathlib import Path
from google.cloud import storage
from dotenv import load_dotenv
import tempfile
from typing import Optional
from pathlib import Path


# Arvix Download to Local Test

In [ ]:
def download_qfin_papers(category="q-fin", max_papers=100, output_dir="./papers"):
    Path(output_dir).mkdir(exist_ok=True)

    client = arxiv.Client(page_size=100, delay_seconds=3)
    search = arxiv.Search(
        query=f"cat:{category}",
        max_results=max_papers,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )

    for result in client.results(search):
        try:
            filename = f"{result.entry_id.split('/abs/')[-1].replace('/', '_')}.pdf"
            result.download_pdf(dirpath=output_dir, filename=filename)
            print(f"✓ {result.title[:60]}...")
        except Exception as e:
            print(f"✗ Failed: {e}")

# Download papers
download_qfin_papers(category="q-fin.MF", max_papers=10)

✓ Estimating the Hurst parameter from the zero vanna implied v...
✓ A mathematical study of the excess growth rate...
✓ The Omniscient, yet Lazy, Investor...
✓ Extended HJB Equation for Mean-Variance Stopping Problem: Va...
✓ Inverse Behavioral Optimization of QALY-Based Incentive Syst...
✓ Goal-based portfolio selection with fixed transaction costs...
✓ Jump risk premia in the presence of clustered jumps...
✓ Portfolio selection with exogenous and endogenous transactio...
✓ Consumption-Investment Problem in Rank-Based Models...
✓ Multivariate Variance Swap Using Generalized Variance Method...


In [ ]:
import random

# List all downloaded PDF files
paper_files = [f for f in os.listdir("./papers") if f.endswith(".pdf")]

# Select a random paper
if paper_files:
    random_paper_path = os.path.join("./papers", random.choice(paper_files))
    print(f"Loading random paper: {random_paper_path}")

    # need a PDF reader librar to extract text content from the PDF file.
else:
    print("No papers found in the ./papers directory.")

Loading random paper: ./papers/2510.17508v1.pdf


# Run on Colab to download Files into GCS storage bucket set up

In [ ]:
#!pip install google-cloud-storage

In [7]:
!pip install gcloud

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached oauth2client-4.1.3-py2.py3-none-any.whl.metadata (1.2 kB)
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=603037 sha256=07a9a106fa988926f9c7455158437dd0adbf76eb2a099b286c06910265c7dffe
  Stored in directory: c:\users\sebastian\appdata\local\pip\cache\wheels\2a\62\75\3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud


In [ ]:
#  from google.colab import auth
#  auth.authenticate_user()

ModuleNotFoundError: No module named 'google.colab'

In [1]:
!gcloud auth application-default login


'gcloud' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!gcloud config set project seraphic-ripple-476917-i5

'gcloud' is not recognized as an internal or external command,
operable program or batch file.


In [13]:
!gcloud auth application-default set-quota-project seraphic-ripple-476917-i5


Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "seraphic-ripple-476917-i5" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [19]:
# Load environment variables from .env file
load_dotenv()

class ArxivGCSDownloader:
    """Download arXiv papers and upload them to Google Cloud Storage"""

    def __init__(self, bucket_name: Optional[str] = None, gcs_folder: str = "papers"):
        """
        Initialize the downloader with GCS credentials

        Args:
            bucket_name: GCS bucket name (defaults to env variable GCS_BUCKET_NAME)
            gcs_folder: Folder path within the bucket (default: "papers")
        """
        # Load GCS configuration from environment
        self.bucket_name = bucket_name or os.getenv("GCS_BUCKET_NAME")
        self.gcs_folder = gcs_folder

        # Initialize GCS client
        # GOOGLE_APPLICATION_CREDENTIALS env var should point to service account JSON
        credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
        if credentials_path and not os.path.exists(credentials_path):
            raise FileNotFoundError(f"Credentials file not found: {credentials_path}")

        self.storage_client = storage.Client()
        self.bucket = self.storage_client.bucket(self.bucket_name)

        print(f"Initialized GCS client for bucket: {self.bucket_name}")
        print(f"Papers will be uploaded to: gs://{self.bucket_name}/{self.gcs_folder}/")

    def download_and_upload_papers(
        self,
        category: str = "q-fin",
        max_papers: int = 100,
        local_temp_storage: bool = False,
        temp_dir: str = "./papers_temp"
    ):
        """
        Download papers from arXiv and upload to GCS

        Args:
            category: arXiv category (e.g., "q-fin.MF")
            max_papers: Maximum number of papers to download
            local_temp_storage: If True, keep local copies in temp_dir
            temp_dir: Directory for temporary local storage
        """
        # Create temp directory if keeping local copies
        if local_temp_storage:
            Path(temp_dir).mkdir(exist_ok=True)

        # Initialize arXiv client
        client = arxiv.Client(page_size=100, delay_seconds=3)
        search = arxiv.Search(
            query=f"cat:{category}",
            max_results=max_papers,
            sort_by=arxiv.SortCriterion.SubmittedDate
        )

        download_count = 0
        upload_count = 0
        error_count = 0

        print(f"\nStarting download for category: {category}")
        print(f"Maximum papers: {max_papers}\n")

        for result in client.results(search):
            try:
                # Generate filename from paper ID
                filename = f"{result.entry_id.split('/abs/')[-1].replace('/', '_')}.pdf"

                # Use temporary directory for download
                with tempfile.TemporaryDirectory() as tmpdirname:
                    # Download paper to temp location
                    temp_path = os.path.join(tmpdirname, filename)
                    result.download_pdf(dirpath=tmpdirname, filename=filename)
                    download_count += 1

                    # Upload to GCS
                    gcs_path = f"{self.gcs_folder}/{filename}"
                    blob = self.bucket.blob(gcs_path)

                    # Check if file already exists
                    if blob.exists():
                        print(f"⊙ Already exists: {result.title[:50]}... (skipping)")
                        continue

                    # Upload with metadata
                    blob.metadata = {
                        'title': result.title,
                        'authors': ', '.join([author.name for author in result.authors]),
                        'published': str(result.published),
                        'arxiv_id': result.entry_id,
                        'categories': ', '.join(result.categories)
                    }

                    blob.upload_from_filename(temp_path)
                    upload_count += 1

                    print(f"✓ Uploaded: {result.title[:50]}...")
                    print(f"  → gs://{self.bucket_name}/{gcs_path}")

                    # Optionally keep local copy
                    if local_temp_storage:
                        local_path = os.path.join(temp_dir, filename)
                        with open(temp_path, 'rb') as src, open(local_path, 'wb') as dst:
                            dst.write(src.read())

            except Exception as e:
                error_count += 1
                print(f"✗ Failed: {result.title[:50] if hasattr(result, 'title') else 'Unknown'}")
                print(f"  Error: {e}")

        # Print summary
        print(f"\n{'='*60}")
        print(f"Download Summary:")
        print(f"  Downloaded: {download_count}")
        print(f"  Uploaded to GCS: {upload_count}")
        print(f"  Errors: {error_count}")
        print(f"  GCS Location: gs://{self.bucket_name}/{self.gcs_folder}/")
        print(f"{'='*60}")

        return {
            'downloaded': download_count,
            'uploaded': upload_count,
            'errors': error_count
        }

    def list_uploaded_papers(self, max_results: int = 10):
        """List papers in the GCS bucket"""
        print(f"\nListing papers in gs://{self.bucket_name}/{self.gcs_folder}/")
        blobs = self.storage_client.list_blobs(
            self.bucket_name,
            prefix=self.gcs_folder,
            max_results=max_results
        )

        for blob in blobs:
            if blob.name.endswith('.pdf'):
                print(f"  • {blob.name}")
                if blob.metadata:
                    print(f"    Title: {blob.metadata.get('title', 'N/A')[:60]}")

# Example usage
if __name__ == "__main__":
    # Initialize downloader (uses environment variables)
    downloader = ArxivGCSDownloader(bucket_name="raw_papers")

    # Download and upload papers
    stats = downloader.download_and_upload_papers(
        category="q-fin.MF",
        max_papers=5,
        local_temp_storage=False  # Set to True to keep local copies
    )

    # List some uploaded papers
    print("\n")
    downloader.list_uploaded_papers(max_results=5)

Initialized GCS client for bucket: raw_papers
Papers will be uploaded to: gs://raw_papers/papers/

Starting download for category: q-fin.MF
Maximum papers: 50

✓ Uploaded: Estimating the Hurst parameter from the zero vanna...
  → gs://raw_papers/papers/2510.26310v1.pdf
✓ Uploaded: A mathematical study of the excess growth rate...
  → gs://raw_papers/papers/2510.25740v1.pdf
✓ Uploaded: The Omniscient, yet Lazy, Investor...
  → gs://raw_papers/papers/2510.24467v1.pdf
✓ Uploaded: Extended HJB Equation for Mean-Variance Stopping P...
  → gs://raw_papers/papers/2510.24128v1.pdf
✓ Uploaded: Inverse Behavioral Optimization of QALY-Based Ince...
  → gs://raw_papers/papers/2510.22518v1.pdf
✓ Uploaded: Goal-based portfolio selection with fixed transact...
  → gs://raw_papers/papers/2510.21650v1.pdf
✓ Uploaded: Jump risk premia in the presence of clustered jump...
  → gs://raw_papers/papers/2510.21297v1.pdf
✓ Uploaded: Portfolio selection with exogenous and endogenous ...
  → gs://raw_papers/pape